# Classification Tulipes / Lys

Pipeline mono-notebook : parsing -> prétraitement -> entraînement -> visualisation.

Un seul fichier Parquet est écrit, juste avant la visualisation.

> **Règles** : DataFrame uniquement · pas de `collect()` / `toPandas()` / `toList()` · Python pur = affichage seulement


## 0 · Session Spark & imports

In [1]:
import sys
import os
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["PATH"] = os.environ["HADOOP_HOME"] + "\\bin;" + os.environ["PATH"]
import io
import struct

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, FloatType, ArrayType, StringType
)

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

spark = (
    SparkSession.builder
    .appName("TulipsLilies")
    .config("spark.sql.files.ignoreCorruptFiles", "true")
    # Mémoire réduite : machine à 8 Go de RAM, éviter de saturer le système
    .config("spark.driver.memory", "1g")
    .config("spark.executor.memory", "1g")
    # Active un vrai traceback Python si l'UDF crash, au lieu d'une erreur Java opaque
    .config("spark.python.worker.faulthandler.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")

Spark version : 4.1.1


## 1 - Chemins & constantes

In [4]:
TRAIN_PATH   = "./data/Train_5/"
TEST_PATH    = "./data/Test_5/"
OUTPUT_PREDS = "./output/predictions/" # seul Parquet écrit
MODEL_PATH   = "./output/model/"
TARGET_SIZE  = (64, 64)

In [5]:
print(os.environ.get("HADOOP_HOME"))
spark.read.format("binaryFile").load(TRAIN_PATH).limit(1).show()

C:\hadoop
+----+----------------+------+-------+
|path|modificationTime|length|content|
+----+----------------+------+-------+
+----+----------------+------+-------+



## 2 - Parsing

In [6]:
TARGET_W, TARGET_H = TARGET_SIZE

def decode_image_bytes(raw_bytes: bytes):
    try:
        from PIL import Image
        img = Image.open(io.BytesIO(raw_bytes)).convert("RGB")
        img = img.resize((TARGET_W, TARGET_H), Image.LANCZOS)
        raw = img.tobytes()
        n = len(raw)
        pixels = list(struct.unpack(f"{n}B", raw))
        return (TARGET_W, TARGET_H, 3, [float(p) for p in pixels])
    except Exception:
        return None

_decode_schema = StructType([
    StructField("width",    IntegerType(), False),
    StructField("height",   IntegerType(), False),
    StructField("channels", IntegerType(), False),
    StructField("pixels",   ArrayType(FloatType()), False),
])

decode_udf = F.udf(decode_image_bytes, _decode_schema)

def parse_images(path):
    raw = (
        spark.read.format("binaryFile")
        .option("recursiveFileLookup", "true")
        .option("pathGlobFilter", "*.{jpg,jpeg,png,JPG,PNG}")
        .load(path)
    )
    return (
        raw
        .select(
            F.regexp_extract(F.col("path"), r"([^/]+)$", 1).alias("image_id"),
            F.regexp_extract(F.col("path"), r"/([^/]+)/[^/]+$", 1).alias("label"),
            F.col("content").alias("raw_bytes"),
        )
        .withColumn("decoded", decode_udf(F.col("raw_bytes")))
        .filter(F.col("decoded").isNotNull())
        .select(
            "image_id", "label",
            F.col("decoded.pixels").alias("pixels"),
        )
    )

train_parsed_df = parse_images(TRAIN_PATH)
test_parsed_df  = parse_images(TEST_PATH)

print(f"Images train : {train_parsed_df.count()}")
print(f"Images test  : {test_parsed_df.count()}")
test_parsed_df.show()

c:\Users\Julien ANTOGNELLI\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyspark\sql\udf.py:134: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)


Images train : 10
Images test  : 10
+----------+-------+--------------------+
|  image_id|  label|              pixels|
+----------+-------+--------------------+
|000139.jpg|tulipes|[180.0, 129.0, 79...|
|000137.jpg|tulipes|[184.0, 155.0, 50...|
|000063.jpg|    lys|[154.0, 2.0, 1.0,...|
|000140.jpg|tulipes|[118.0, 102.0, 84...|
|000061.jpg|    lys|[131.0, 119.0, 82...|
|000138.jpg|tulipes|[81.0, 15.0, 54.0...|
|000062.jpg|    lys|[205.0, 198.0, 16...|
|000064.jpg|    lys|[118.0, 145.0, 10...|
|000141.jpg|tulipes|[0.0, 72.0, 0.0, ...|
|000065.jpg|    lys|[164.0, 190.0, 21...|
+----------+-------+--------------------+



## 3 - Prétraitement 

On garde `pixels` (RGB brut, 0-255) intact pour l'affichage futur dans Streamlit,
et on ajoute une colonne `pixels_gray` : niveaux de gris normalisés en [0.0, 1.0].

Conversion RGB -> nuances de gris : formule de luminance pondérée (standard) :
```
gray = 0.299*R + 0.587*G + 0.114*B
```

`pixels` est une liste aplatie `[R,G,B, R,G,B, ...]` de taille 64*64*3 = 12288.
L'UDF regroupe les valeurs par 3, applique la formule et renvoie les pixels en niveaux de gris

In [8]:
def rgb_to_grayscale(pixels):
    if pixels is None:
        return None
    gray = []
    for i in range(0, len(pixels), 3):
        r, g, b = pixels[i], pixels[i + 1], pixels[i + 2]
        # formule standard de luminance perceptuelle
        g_value = 0.299 * r + 0.587 * g + 0.114 * b
        gray.append(g_value)
    return gray

gray_udf = F.udf(rgb_to_grayscale, ArrayType(FloatType()))

def preprocess(df):
    return df.withColumn("pixels_grayscale", gray_udf(F.col("pixels")))

train_preprocessed_df = preprocess(train_parsed_df)
test_preprocessed_df  = preprocess(test_parsed_df)

print("Aperçu après prétraitement :")
train_preprocessed_df.select("image_id", "label", "pixels_grayscale").show(5, truncate=40)

# Vérification rapide : taille attendue = 64*64 = 4096 valeurs en niveaux de gris
expected_len = TARGET_W * TARGET_H
check_len = (
    train_preprocessed_df
    .select(F.size(F.col("pixels_grayscale")).alias("len"))
    .first()["len"]
)
print(f"Taille pixels_grayscale (attendu {expected_len}) : {check_len}")

Aperçu après prétraitement :
+----------+-------+----------------------------------------+
|  image_id|  label|                        pixels_grayscale|
+----------+-------+----------------------------------------+
|000005.jpg|    lys|[30.616, 32.018, 33.491, 35.263, 37.9...|
|000004.jpg|tulipes|[93.374, 96.787, 99.684, 102.755, 104...|
|000004.jpg|    lys|[207.335, 205.221, 205.107, 205.107, ...|
|000002.jpg|    lys|[135.315, 136.201, 135.087, 136.087, ...|
|000001.jpg|    lys|[22.399, 18.274, 60.137, 102.701, 108...|
+----------+-------+----------------------------------------+
only showing top 5 rows
Taille pixels_grayscale (attendu 4096) : 4096


In [ ]:
def rgb_to_normalized_gray(pixels):
    if pixels is None:
        return None
    gray = []
    for i in range(0, len(pixels), 3):
        r, g, b = pixels[i], pixels[i + 1], pixels[i + 2]
        # formule standard de luminance perceptuelle
        g_value = 0.299 * r + 0.587 * g + 0.114 * b
        # normalise la valeur résultante dans l'intervalle [0, 1] au lieu de [0, 255],
        gray.append(g_value / 255.0)
    return gray

gray_udf = F.udf(rgb_to_normalized_gray, ArrayType(FloatType()))

def preprocess(df):
    return df.withColumn("pixels_gray", gray_udf(F.col("pixels")))

train_preprocessed_df = preprocess(train_parsed_df)
test_preprocessed_df  = preprocess(test_parsed_df)

print("Aperçu après prétraitement :")
train_preprocessed_df.select("image_id", "label", "pixels_gray").show(5, truncate=40)

# Vérification rapide : taille attendue = 64*64 = 4096 valeurs en niveaux de gris
expected_len = TARGET_W * TARGET_H
check_len = (
    train_preprocessed_df
    .select(F.size(F.col("pixels_gray")).alias("len"))
    .first()["len"]
)
print(f"Taille pixels_gray (attendu {expected_len}) : {check_len}")

## ML couleurs - prétraitement en bytes

## ML couleurs normalisées


## ML grayscale

## ML grayscale normalisé